# Scrape antifakenewscenter.com – #มิจฉาชีพ news

Keep only posts tagged **#มิจฉาชีพ** together with **#อาชญากรรมออนไลน์** and/or **#ข่าวปลอม**, then
label them. Run the cells from top to bottom; the last cell saves the dataset. Everything is in this
notebook: the scraper, the LLM labeller and its prompts (only the organization list comes from
`build_organizations.ipynb`).

The site is WordPress, so instead of parsing the tag page HTML we read the same posts of the
`/tag/มิจฉาชีพ/` archive through its public REST API (`/wp-json/wp/v2/posts?tags=<id>`). Each post
already carries its tag IDs, so no per-post page fetch is needed.

Output columns, in order:

| Columns | Made by |
|---|---|
| `post_id`, `date`, `hashtag`, `title`, `content` | the scrape (`date` is `DD/MM/YYYY` in the CSV) |
| `Channel` | keyword rule: `call` if the scammer phones the victim, otherwise `text` |
| `ORG` | keyword match against a list of Thai organizations (see "Organization column") |
| `scam_type`, `method_channel` | the LLM, pass 1 (see "Scam type and channel columns") |
| `call_suitable`, `summary`, `caller_role`, `goal`, `tactics`, `phrases`, `scenario_key`, `scenario_group` | the LLM, pass 2 (see "Scenario columns") |
| `source_url` | built from `post_id` |

`Channel` and `ORG` come from keyword rules, so they are a first pass, not ground truth.

The last cell saves everything to `data/antifakenews_scam.csv` (or `data/antifakenews_scam_test.csv`
when `MAX_POSTS` is set for a test run).

In [ ]:
import html
import io
import re
import time
from collections import Counter
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

API = "https://www.antifakenewscenter.com/wp-json/wp/v2"
HEADERS = {"User-Agent": "Mozilla/5.0 (research scraper; Callas project)"}
DELAY = 1.0  # seconds between requests

BASE_TAG = "มิจฉาชีพ"
TARGET_TAGS = ["อาชญากรรมออนไลน์", "ข่าวปลอม"]

# Test run: a number (e.g. 10) fetches only the first page of the API and keeps that many of the
# newest posts, and saves to a separate file so the real dataset is not overwritten.
# None = the full run: every page, every post, saved to data/antifakenews_scam.csv.
MAX_POSTS = None
OUT_PATH = Path("data/antifakenews_scam.csv" if MAX_POSTS is None else "data/antifakenews_scam_test.csv")

In [ ]:
def get_json(path, params, retries=3):
    """GET an API endpoint with retries; returns (json, response headers)."""
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(f"{API}/{path}", params=params, headers=HEADERS, timeout=30)
            r.raise_for_status()
            return r.json(), r.headers
        except requests.RequestException:
            if attempt == retries:
                raise
            time.sleep(2 * attempt)


def find_tag_id(name):
    """Resolve a tag name (with or without a leading '#') to its WordPress tag ID."""
    tags, _ = get_json("tags", {"search": name, "per_page": 100, "_fields": "id,name,count"})
    exact = [t for t in tags if t["name"].lstrip("#").strip() == name]
    if not exact:
        raise ValueError(f"tag not found: {name}")
    return max(exact, key=lambda t: t["count"])["id"]


tag_ids = {name: find_tag_id(name) for name in [BASE_TAG, *TARGET_TAGS]}
tag_ids

In [ ]:
def fetch_posts(tag_id, max_pages=None):
    """All posts of a tag (same set as the /tag/<slug>/ archive pages), newest first.
    max_pages stops early: 1 page is 100 posts, enough for a test run."""
    posts, page, total_pages = [], 1, 1
    while page <= total_pages and (max_pages is None or page <= max_pages):
        batch, headers = get_json("posts", {"tags": tag_id, "per_page": 100, "page": page,
                                            "_fields": "id,date,title,content,tags"})
        total_pages = int(headers["X-WP-TotalPages"])
        posts += batch
        print(f"page {page}/{total_pages}: {len(posts)}/{headers['X-WP-Total']} posts")
        page += 1
        time.sleep(DELAY)
    return posts


raw_posts = fetch_posts(tag_ids[BASE_TAG], max_pages=None if MAX_POSTS is None else 1)

In [ ]:
FOOTER_HEADING = "ช่องทางการติดตามและแจ้งเบาะแสข่าวปลอม"
FOOTER_START = "LINE : @antifakenewscenter"  # the site's contact block closes every post from here on

# The scammer phones the victim -> "call", anything else -> "text".
# Only phrases about being called count; a bare "โทรศัพท์" or a report hotline does not.
CALL = (r"คอลเซ็นเตอร์|ทางโทรศัพท์|ผ่านโทรศัพท์|(?i:\bcall\s*center\b)|รับสาย|ได้รับสาย"
        r"|โทรศัพท์(?:มา|หา|ติดต่อ|สอบ|แจ้ง|ถาม|ติดตาม)"
        r"|โทร(?:มา|หา|ไป|แจ้ง|ติดต่อ|สอบ|ถาม|ตรวจสอบ|หลอก|ขู่|เข้า)")
HOTLINE = r"(?i:call\s*center)\s*\d{3,4}|สายด่วน\s*\d+"  # e.g. "Call Center 1284", not a scam call


def clean_html(fragment):
    soup = BeautifulSoup(fragment, "html.parser")
    for tag in soup(["script", "style", "iframe", "figure", "img"]):
        tag.decompose()
    lines = (re.sub(r"[ \t]+", " ", line).strip() for line in soup.get_text("\n").replace("\xa0", " ").splitlines())
    lines = [line for line in lines if line]
    # drop the contact footer (LINE, website, Facebook, X, TikTok, IG) and its heading
    for i, line in enumerate(lines):
        if line.startswith(FOOTER_START):
            lines = lines[:i]
            if lines and lines[-1] == FOOTER_HEADING:
                lines.pop()
            break
    return "\n".join(lines)


def label_channel(text):
    return "call" if re.search(CALL, re.sub(HOTLINE, "", text)) else "text"


rows = []
for p in raw_posts:  # all fetched under #มิจฉาชีพ; keep those that also carry a target hashtag
    matched = [name for name in TARGET_TAGS if tag_ids[name] in p["tags"]]
    if matched:
        rows.append({
            "post_id": p["id"],
            "date": p["date"],
            "hashtag": ", ".join(matched),  # both tags -> one row, e.g. "อาชญากรรมออนไลน์, ข่าวปลอม"
            "title": html.unescape(BeautifulSoup(p["title"]["rendered"], "html.parser").get_text()).strip(),
            "content": clean_html(p["content"]["rendered"]),
        })

df = pd.DataFrame(rows)
# real datetime64 column, so it sorts chronologically (a DD/MM/YYYY string would sort by day first)
df["date"] = pd.to_datetime(df["date"]).dt.normalize()
df["Channel"] = (df["title"] + "\n" + df["content"]).map(label_channel)
df = df.sort_values("date", ascending=False, ignore_index=True)
df

In [ ]:
# drop posts with no title or no content (e.g. image-only posts): there is nothing to label in them
empty = (df["title"].str.strip() == "") | (df["content"].str.strip() == "")
if empty.any():
    print(f"dropping {empty.sum()} posts with no title or content: {df.loc[empty, 'post_id'].tolist()}")
df = df[~empty].reset_index(drop=True)

# test run: keep only the newest MAX_POSTS posts (df is sorted newest first)
if MAX_POSTS is not None:
    df = df.head(MAX_POSTS)
    print(f"test run: kept the {len(df)} newest posts")

# check the scrape now, before ORG and LLM labelling, so a bad scrape stops here and costs no API calls
assert df["post_id"].is_unique
assert df["date"].notna().all()
assert df["hashtag"].str.contains("|".join(TARGET_TAGS)).all()
assert not df["content"].str.contains(FOOTER_START, regex=False).any()
print(f"{len(df)} posts passed the checks")

## Organization column

The `ORG` column names the Thai organization each post says the scammer pretended to be. It is
matched against `data/thai_organizations.csv`, a list of Thai government agencies, ministries and
banks with their **full names and abbreviations**. That list is built by `build_organizations.ipynb`
(from DGA open data, Thai Wikipedia and Wikidata); run that notebook first, and again only when you
want a fresh list.

A post is matched on any alias of an
organization (full name, abbreviation, `ธ.`/`แบงก์` + bank name, English name) after removing spaces,
because Thai spacing is irregular. The title is searched first. If it names no organization, the most
mentioned one in the content is used, ignoring the police, since almost every post tells the reader to
report to them. Limits: private companies (HomePro, Lotus's, Singha ...) and local bodies are not in
the list, so those posts get an empty `ORG`, and a match means the post *mentions* the
organization, not strictly that the scammer impersonated it.

In [ ]:
# the organization list is built by build_organizations.ipynb; here it is only read
# (dtype=str and keep_default_na=False so an empty abbreviation stays "" rather than NaN)
ORG_PATH = Path("data/thai_organizations.csv")
orgs = pd.read_csv(ORG_PATH, dtype=str, encoding="utf-8-sig", keep_default_na=False)

has_thai = lambda s: bool(re.search(r"[฀-๿]", s))  # a Thai alias needs more characters than an English one

print(f"{len(orgs)} organizations from {ORG_PATH}; {(orgs['abbr_en'] != '').sum()} with an English abbreviation")

In [ ]:
def squash(text):
    """Drop Thai spacing (it is irregular) but keep the gap between two Latin words."""
    text = text.replace("​", "")

    def gap(m):
        before = text[m.start() - 1] if m.start() else ""
        after = text[m.end()] if m.end() < len(text) else ""
        return " " if re.match(r"[A-Za-z0-9]", before) and re.match(r"[A-Za-z0-9]", after) else ""

    return re.sub(r"\s+", gap, text)


# colloquial names the lists do not carry: canonical Thai name -> extra aliases
MANUAL_ALIASES = {
    "สำนักงานตำรวจแห่งชาติ": ["ตำรวจ", "ตร."],
    "กรมสอบสวนคดีพิเศษ": ["ดีเอสไอ"],
    "ธนาคารแห่งประเทศไทย": ["แบงก์ชาติ"],
    "ธนาคารทหารไทยธนชาต": ["ทีทีบี"],
}
# ordinary words that must not stand in for an organization on their own
AMBIGUOUS_BANK_BARE = {"กรุงเทพ", "ไทย", "เครดิต"}  # keep "ธนาคาร"/"ธ." in front
GENERIC_COMPANY_BASE = {"ขนส่ง", "การบิน"}          # "บริษัท ขนส่ง จำกัด" minus บริษัท/จำกัด
BLOCKED_ALIASES = {"SME"}                           # loan product word, not the bank
IGNORE_IN_CONTENT = {"สำนักงานตำรวจแห่งชาติ"}        # posts end with "report to the police"


def org_aliases(row):
    name = row.name_th
    base = re.sub(r"\s*\((?:มหาชน|องค์การมหาชน)\)|\s*จำกัด|^บริษัท\s*", "", name).strip()
    out = {name, *MANUAL_ALIASES.get(name, []), *filter(None, row.abbr_en.split("|"))}
    if len(squash(base)) >= 6 and base not in GENERIC_COMPANY_BASE:
        out.add(base)
    if base.startswith("ธนาคาร"):
        rest = base[len("ธนาคาร"):].strip()
        out |= {"ธ." + rest, "แบงก์" + rest}
        if len(rest) >= 4 and rest not in AMBIGUOUS_BANK_BARE:
            out.add(rest)
    for a in filter(None, row.abbr_th.split("|")):
        out |= {a, re.sub(r"^สำนักงาน\s*", "", a)}  # "สำนักงาน ก.ล.ต." is also written "ก.ล.ต."
    # Thai abbreviations need >= 4 characters incl. dots, English ones >= 3, or they hit ordinary words
    return {a for a in out if a not in BLOCKED_ALIASES and len(squash(a)) >= (4 if has_thai(a) else 3)}


ALIAS_TO_ORG = {}
for row in orgs.itertuples():
    for alias in org_aliases(row):
        ALIAS_TO_ORG.setdefault(squash(alias), row.name_th)  # on a clash the first listed organization wins

# one pattern, longest alias first so the longest name wins; English aliases must be whole words
ALIAS_RE = re.compile("|".join(
    re.escape(a) if has_thai(a) else rf"(?<![A-Za-z]){re.escape(a)}(?![A-Za-z])"
    for a in sorted(ALIAS_TO_ORG, key=len, reverse=True)
))


def org_mentions(text):
    """Organizations named in the text, in order of appearance."""
    return [ALIAS_TO_ORG[m.group()] for m in ALIAS_RE.finditer(squash(text))]


def label_organization(title, content):
    in_title = dict.fromkeys(org_mentions(title))  # unique, in order
    if in_title:
        return ", ".join(in_title)
    counts = Counter(org for org in org_mentions(content) if org not in IGNORE_IN_CONTENT)
    return counts.most_common(1)[0][0] if counts else ""


df["ORG"] = [label_organization(t, c) for t, c in zip(df["title"], df["content"])]
labelled = df.loc[df["ORG"] != "", "ORG"]
print(f"{len(labelled)}/{len(df)} posts have an organization")
labelled.str.split(", ").explode().value_counts().head(20)

## LLM labelling engine

The next two sections label the posts with the LLM on the project's vLLM server, in two passes:
first the scam type and channel, then the scenario columns. The cell below holds the code both
passes share.

- **Connection**: `VLLM_BASE_URL`, `VLLM_API_KEY` and `VLLM_MODEL` from the repo-root `.env`, the
  same settings the rest of Callas uses. Thinking is turned off so the tokens go to the answer.
- **Batches**: posts are sent several at a time, each tagged with its `post_id`, e.g. `[78875]`; the
  model answers `{"posts": [{"post_id": 78875, ...}, ...]}` and the ids map the answers back to the
  rows. A bad or cut-off reply splits the batch in half; posts missing from a reply are asked once
  more; posts that still fail get empty fields and `llm_error = True`.
- **Checkpoints**: every finished answer is appended to `data/llm_<pass>_checkpoint.jsonl`, so a run
  that stops early resumes where it left off. The file is deleted once every post is labelled.
  Delete it by hand after editing a prompt, or the old answers are reused.

Each pass keeps its own prompts in its section, as Python strings. `{name}` placeholders in them are
filled with `str.replace`, so braces inside a post cannot break the prompt.

In [ ]:
import json
import os
import sys
from dataclasses import dataclass
from typing import Callable

import yaml
from dotenv import load_dotenv
from openai import OpenAI

REPO = Path.cwd().parent  # this notebook runs from scraping/
load_dotenv(REPO / ".env")  # VLLM_BASE_URL, VLLM_API_KEY, VLLM_MODEL; variables already set in the shell win

# the call simulator's slot names ({MONEY}, {URL} ...), so the labels only use slots the simulator knows
sys.path.insert(0, str(REPO / "modules"))
from data_pipeline.config import SLOT_NAMES  # noqa: E402

MAX_CONTENT_CHARS = 3000  # long fact-check posts state the claim first, so the start is enough


def fill(template, **values):
    """Replace {key} placeholders with str.replace, so braces in a post cannot break the prompt."""
    for key, value in values.items():
        template = template.replace("{" + key + "}", value)
    return template


@dataclass
class Task:
    """One labelling pass: its prompts, the fields it answers and how one answer is cleaned."""

    name: str  # also names the checkpoint file
    system_prompt: str  # built once, the same for every post
    user_prompt: str  # one post; its {placeholders} are the lower-cased column names
    columns: list  # dataset columns put into the user prompt
    fields: list  # answer fields
    clean: Callable  # raw answer item -> {field: text}
    tokens_per_post: int  # answer budget per post
    batch_size: int  # posts per request

    @property
    def checkpoint(self):
        return Path("data") / f"llm_{self.name}_checkpoint.jsonl"


def get_client():
    """Client for the vLLM server in .env. The timeout is long because a batch of posts makes a long
    answer; VLLM_TIMEOUT_S is meant for the short replies of the live call, so it is not used here."""
    return OpenAI(base_url=os.environ["VLLM_BASE_URL"], api_key=os.environ.get("VLLM_API_KEY") or "EMPTY",
                  timeout=600, max_retries=2)


def ask(client, task, user_text, n_posts):
    """One chat call. Asks for JSON mode, and drops it if the server does not support it."""
    request = dict(
        model=os.environ["VLLM_MODEL"],
        messages=[{"role": "system", "content": task.system_prompt}, {"role": "user", "content": user_text}],
        temperature=0,
        max_tokens=task.tokens_per_post * n_posts,  # the answer grows with the number of posts
        # Qwen 3.x thinks before answering; turn it off so the tokens go to the JSON answer
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    try:
        reply = client.chat.completions.create(**request, response_format={"type": "json_object"})
    except Exception as error:  # some servers reject response_format; retry without it
        if "response_format" not in str(error).lower():
            raise
        reply = client.chat.completions.create(**request)
    choice = reply.choices[0]
    if not choice.message.content:
        raise ValueError(f"empty reply (finish_reason={choice.finish_reason})")
    if choice.finish_reason == "length":  # hit max_tokens, so the JSON is cut off
        raise ValueError("reply cut off at max_tokens")
    return choice.message.content


def parse_reply(task, raw):
    """Turn the model's reply into {post_id: answer} for the posts it answered."""
    match = re.search(r"\{.*\}", raw, re.DOTALL)  # tolerate text or <think> blocks around the JSON
    if not match:
        raise ValueError(f"no JSON in reply: {raw[:300]!r}")
    items = json.loads(match.group(0)).get("posts", [])
    return {str(item.get("post_id", "")).strip(): task.clean(item) for item in items}


def prompt_values(post):
    """A post's values as prompt text, with the content cut to MAX_CONTENT_CHARS."""
    values = {key: str(value) for key, value in post.items()}
    values["content"] = values["content"][:MAX_CONTENT_CHARS]
    return values


def label_batch(client, task, posts, retry=True):
    """Label a list of posts (dicts of the task's columns, lower-cased) in one request.
    Returns {post_id: answer}.

    - Bad or cut-off reply (ValueError): the batch is split in half and each half is asked on its
      own, so one broken reply does not lose the answers of the whole batch.
    - Connection error, timeout or rate limit (the client already retried twice): no answers.
    Posts still without an answer are asked once more (retry); after that they get empty fields
    and llm_error=True.
    """
    # one user message holding all the posts, each tagged with its post_id
    message = "\n\n".join(fill(task.user_prompt, **prompt_values(post)) for post in posts)
    try:
        answers = parse_reply(task, ask(client, task, message, len(posts)))
    except ValueError as error:
        if len(posts) > 1:
            print(f"bad reply for {len(posts)} posts ({error}); splitting the batch")
            half = len(posts) // 2
            return {**label_batch(client, task, posts[:half], retry),
                    **label_batch(client, task, posts[half:], retry)}
        print(f"bad reply: {error}")
        answers = {}
    except Exception as error:
        print(f"request failed: {error}")
        answers = {}

    results = {post["post_id"]: {**answers[str(post["post_id"])], "llm_error": False}
               for post in posts if str(post["post_id"]) in answers}
    missing = [post for post in posts if post["post_id"] not in results]
    if missing and retry:
        results.update(label_batch(client, task, missing, retry=False))  # ask once more for the posts left out
    else:
        for post in missing:
            print(f"failed: [{post['post_id']}] {str(post['title'])[:60]}")
            results[post["post_id"]] = {**dict.fromkeys(task.fields, ""), "llm_error": True}
    return results


def load_checkpoint(task):
    """Answers saved by an earlier run that stopped early, as {post_id (str): answer}."""
    if not task.checkpoint.exists():
        return {}
    saved = {}
    for line in task.checkpoint.read_text(encoding="utf-8").splitlines():
        try:
            item = json.loads(line)
            saved[item["post_id"]] = item["answer"]
        except (ValueError, KeyError):  # a line half-written when the run was killed
            pass
    return saved


def label_all(df, task):
    """Label every row of df with one task. Returns the task's fields + llm_error, same index."""
    client = get_client()
    labels = load_checkpoint(task)  # keyed by str(post_id), so int and str ids both match
    if labels:
        print(f"resuming: {len(labels)} answers found in {task.checkpoint.name}")
    # one dict per post, keys = the task's columns lower-cased ("ORG" -> {org} in the prompt)
    rows = df[task.columns].rename(columns=str.lower).to_dict("records")
    posts = [post for post in rows if str(post["post_id"]) not in labels]

    task.checkpoint.parent.mkdir(parents=True, exist_ok=True)
    with task.checkpoint.open("a", encoding="utf-8") as checkpoint:
        for start in range(0, len(posts), task.batch_size):
            for post_id, answer in label_batch(client, task, posts[start:start + task.batch_size]).items():
                labels[str(post_id)] = answer
                if not answer["llm_error"]:  # failed posts are not saved, so the next run asks again
                    checkpoint.write(json.dumps({"post_id": str(post_id), "answer": answer}, ensure_ascii=False) + "\n")
            checkpoint.flush()  # on disk now, even if the next batch crashes
            print(f"labelled {min(start + task.batch_size, len(posts))}/{len(posts)}")

    # put the answers back in the order of df's rows, matched by post_id
    result = pd.DataFrame([labels[str(post_id)] for post_id in df["post_id"]], index=df.index)
    if not result["llm_error"].any():
        task.checkpoint.unlink()  # everything labelled; the next run starts fresh
    return result


print(f"vLLM server: {os.environ.get('VLLM_BASE_URL')} | model: {os.environ.get('VLLM_MODEL')}")

## Scam type and channel columns (labelled by the LLM, pass 1)

Two columns are filled by the first pass, on every post, 50 posts per request:

- `scam_type`: one of the five Callas scam types (`fake_authority`, `romance`, `investment`,
  `phishing_otp`, `lottery`), or empty when the post describes no scam (an overview of several types,
  a fact-check of a rumour).
- `method_channel`: where the scam happens (see "Channel column" below).

The first cell holds the prompts: the scam type definitions (sent whole as part of the instructions,
so a change there changes the labels), the system prompt and the template for one post. The second
cell checks each answer and sets up the pass; the third runs it.

In [ ]:
# Prompts of pass 1. SCAM_TYPE_DEFINITIONS goes into {definitions} of LABEL_SYSTEM;
# LABEL_USER is one post, and all posts of a batch are joined with a blank line into one message.

SCAM_TYPE_DEFINITIONS = """# Scam type definitions

How the `scam_type` column in `data/antifakenews_scam.csv` was labelled. The five types are the
Callas scam types, with wider definitions so that every scam on antifakenewscenter.com fits one.

This text is sent to the model as its instructions, so a change here changes the labels.

## Types

| Type | Definition |
|---|---|
| `fake_authority` | Pretends to be a trusted organization or person in an official role (government agency, police, bank, utility, post or courier, company or shop), often through a fake page or account in its name. It threatens the victim, or offers a service such as a loan, a driving licence, money recovery or a job, or a government handout, welfare, stimulus or refund payment in an official's name, then demands a fee, fine, transfer, or personal data such as an ID card or bank account number. |
| `romance` | Builds trust as a partner, friend, relative or acquaintance, or takes over or copies the account of someone the victim knows or a celebrity, then asks to borrow money or for a transfer. |
| `investment` | Promises high or guaranteed returns from stocks, crypto, trading, funds or paid "task" jobs. Shows fake profits, then blocks withdrawals until more "fees" are paid. |
| `phishing_otp` | Pretends to be a bank, courier, company or official, and the main ask is an OTP, password or PIN, or to click a link (SMS, QR code, fake website) or install an app from one, including remote-control apps. A request only for personal data, an ID card or a bank account number, with no code or link, is `fake_authority`. |
| `lottery` | Claims the victim won a prize, gift, lucky draw or lottery tip, from a company, shop, draw or celebrity, then asks for a fee or bank details before they can receive it. A government handout, welfare or stimulus payment offered in an official's name is `fake_authority`, not `lottery`. |

## Tie-break

When a post fits several types, the first match in this order wins:

1. `romance`: the scam relies on a personal relationship or trust in someone the victim knows.
2. `investment`: the lure is profit or returns.
3. `lottery`: the lure is a prize, a draw or something free from a company or draw (not a government payment).
4. `phishing_otp`: the main ask is an OTP, password or PIN, a link or an app.
5. `fake_authority`: everything else that uses impersonation to get paid.

## Channel field

`method_channel` says where the scam happens, from a fixed list, comma separated
(e.g. post 78875: เพจเฟซบุ๊ก).

Channel values: เพจเฟซบุ๊ก, กลุ่มเฟซบุ๊ก, Messenger, TikTok, LINE, SMS, โทรศัพท์, เว็บไซต์, อีเมล, เอกสาร,
แอปพลิเคชัน, โซเชียลมีเดีย (social media, platform not named), พอดแคสต์, QR Code, Google Form."""

LABEL_SYSTEM = """You label Thai scam-warning posts from antifakenewscenter.com. You get many posts at once, each
starting with its post_id in square brackets, e.g. [78875]. Label every post on its own.

{definitions}

Rules:
- Pick exactly one scam_type with the definitions and tie-break above. Use "" (empty) when the post
  describes no scam, e.g. an overview of several scam types or a fact-check of a rumour.
- method_channel: only values from the channel list, several joined with ", ".
- Never invent a detail the post does not state; leave that field "".
- When scam_type is "", method_channel is "".

Reply with JSON only, no other text: one object with a "posts" list that has one item for every
post, in order, where "post_id" is the id from the post's square brackets:
{"posts": [{"post_id": 78875, "scam_type": "", "method_channel": ""}]}"""

LABEL_USER = """[{post_id}]
Title: {title}
Content: {content}"""

In [ ]:
SCAM_TYPES = {"fake_authority", "romance", "investment", "phishing_otp", "lottery"}
CHANNELS = {
    "เพจเฟซบุ๊ก", "กลุ่มเฟซบุ๊ก", "Messenger", "TikTok", "LINE", "SMS", "โทรศัพท์", "เว็บไซต์", "อีเมล",
    "เอกสาร", "แอปพลิเคชัน", "โซเชียลมีเดีย", "พอดแคสต์", "QR Code", "Google Form",
}
FIELDS = ["scam_type", "method_channel"]


def clean(item):
    """Keep only allowed values for one post's scam type answer."""
    out = {field: str(item.get(field) or "").strip() for field in FIELDS}

    if out["scam_type"] not in SCAM_TYPES:
        out["scam_type"] = ""
    channels = [c.strip() for c in out["method_channel"].split(",")]
    out["method_channel"] = ", ".join(c for c in channels if c in CHANNELS)
    if not out["scam_type"]:  # no scam -> no method
        out["method_channel"] = ""
    return out


LABEL_TASK = Task(
    name="labels",
    system_prompt=fill(LABEL_SYSTEM, definitions=SCAM_TYPE_DEFINITIONS),  # the definitions go in whole
    user_prompt=LABEL_USER,
    columns=["post_id", "title", "content"],
    fields=FIELDS,
    clean=clean,
    tokens_per_post=100,  # a scam type and a few channel names
    batch_size=50,
)

In [ ]:
# 50 posts per API call, one call after another; returns scam_type and method_channel + llm_error
# (a run that stops early resumes from data/llm_labels_checkpoint.jsonl)
labels = label_all(df, LABEL_TASK)
df[FIELDS] = labels[FIELDS]

print(f"{labels['llm_error'].sum()} posts failed to label")
df["scam_type"].replace("", "(none)").value_counts()

## Channel column

`method_channel` says where each scam happens: เพจเฟซบุ๊ก, กลุ่มเฟซบุ๊ก, TikTok, LINE, SMS, โทรศัพท์,
เว็บไซต์, อีเมล, เอกสาร ... (comma separated). It comes from the same LLM call as `scam_type` (cells
above); channels outside the fixed list are dropped, and posts with no `scam_type` have it empty.
A post whose `method_channel` lists โทรศัพท์ is a phone scam (`call_suitable` = `call`, see "Scenario columns").

In [ ]:
df[["title", "method_channel"]].head()

## Scenario columns (labelled by the LLM, pass 2)

A second pass, only on posts with a `scam_type`, 20 posts per request, describes each scam the way
the call simulator (`modules/data_pipeline`) needs it:

| Column | What it holds |
|---|---|
| `call_suitable` | `call` (the scammer phones; always `call` when `method_channel` lists โทรศัพท์), `convertible` (could be told in a call) or `no` |
| `summary` | one English line naming the pattern |
| `caller_role` | who the scammer claims to be, in English |
| `goal` | what the scammer wants by the end, in English, with slots such as `{MONEY}` |
| `tactics` | the scam's steps in order, in English, joined with ` \| ` |
| `phrases` | Thai lines the post quotes from the scammer, with slots, joined with ` \| ` |
| `scenario_key` | the matching scenario in the simulator's `scenarios.yaml`, or `new` |
| `scenario_group` | a short name shared by posts with the same new situation |
| `source_url` | the post's link, built from `post_id` (no LLM) |

The first cell holds the prompts (`{scenarios}` is filled with the simulator's existing scam
scenarios, `{slots}` with its slot names). The second cell checks each answer: real numbers, links
and LINE IDs the model left in are turned into the simulator's slots. The third runs the pass. Posts
with no `scam_type` have these columns empty (except `source_url`).

In [ ]:
# Prompts of pass 2. {scenarios} in SCENARIO_SYSTEM is filled with the simulator's existing scam
# scenarios (scenarios.yaml), {slots} with its slot names. SCENARIO_USER is one post ({org} is the
# ORG column, empty when no listed organization was found).

SCENARIO_SYSTEM = """You read Thai scam-warning posts from antifakenewscenter.com and describe each scam so that a
phone-call simulator can imitate it to train a scam-detection system. You get many posts at once,
each starting with its post_id in square brackets, e.g. [78875], then its scam_type, organization,
title and content. Describe every post on its own.

Fields for each post:
- call_suitable: "call" if the post says the scammer phones or voice-calls the victim (a LINE
  or video call counts). "convertible"
  if it is not a phone call but the same story could be told in one (an official, a bill, a
  parcel, a prize, a relative asking for help). "no" if the scam depends on something seen or
  read (a fake page or ad, a QR code, a document, a shop listing, a fake website alone).
- summary: one short English line naming the pattern, e.g. "Caller poses as the post office and
  asks for a redelivery fee".
- caller_role: who the scammer claims to be, in English, e.g. "officer of the electricity authority".
- goal: one English sentence: what the scammer wants the victim to do by the end, with slots for
  amounts, accounts, links, codes and phone numbers, e.g. "get the receiver to pay {MONEY} to
  {BANK_ACCOUNT}".
- tactics: a list of 2 to 5 short English steps, in the order the scam goes.
- phrases: a list of Thai lines the post quotes as said or written by the scammer (in quotes, or
  after อ้างว่า / แจ้งว่า / บอกว่า). Copy the post's words exactly, only replacing numbers, amounts,
  links, account numbers and names with slots. Use [] when the post quotes nothing.
- scenario_key: the key of the existing scenario below with the same scam_type and the same
  situation (the same kind of caller asking for the same kind of thing), or "new".
- scenario_group: when scenario_key is "new", a short snake_case English name for the situation,
  e.g. "post_redelivery_fee"; give posts with the same situation the same name. Otherwise repeat
  the scenario_key.

Existing scenarios (scam_type / key: role; wants: goal):
{scenarios}

Rules:
- Write in English, except phrases, which stay in Thai.
- Never write a real phone number, account number, amount, link, LINE ID or person's name: use a
  slot instead. The only slots are: {slots}
- Never invent a detail the post does not state; leave that field "" or [].

Reply with JSON only, no other text: one object with a "posts" list that has one item for every
post, in order, where "post_id" is the id from the post's square brackets:
{"posts": [{"post_id": 78875, "call_suitable": "", "summary": "", "caller_role": "", "goal": "", "tactics": [], "phrases": [], "scenario_key": "new", "scenario_group": ""}]}"""

SCENARIO_USER = """[{post_id}]
scam_type: {scam_type}
Organization: {org}
Title: {title}
Content: {content}"""

In [ ]:
SCENARIO_FIELDS = [
    "call_suitable", "summary", "caller_role", "goal", "tactics", "phrases", "scenario_key", "scenario_group",
]
SUITABLE = {"call", "convertible", "no"}
LIST_SEP = " | "  # tactics and phrases are lists; a csv cell holds them joined with this
SCENARIOS_YAML = REPO / "modules" / "data_pipeline" / "config" / "scenarios.yaml"
POST_URL = "https://www.antifakenewscenter.com/?p={}"

# real numbers, links and LINE IDs the model left in its answer -> the simulator's slots
TO_SLOTS = [
    # a link; [\w-]+(?:\.[\w-]+)* takes every part of the domain, e.g. thaipoliceonline.go.th
    (re.compile(r"https?://\S+|www\.\S+|\b[\w-]+(?:\.[\w-]+)*\.(?:com|net|org|co|ly|me|th|info|xyz)\b\S*"), "{URL}"),
    (re.compile(r"[\w-]+\.\{URL\}"), "{URL}"),  # a domain an earlier run cut in half: thaipoliceonline.{URL}
    (re.compile(r"\d[\d,.]*\s*(?:บาท|baht)", re.IGNORECASE), "{MONEY}"),  # 50,000 บาท
    (re.compile(r"\b0\d{1,2}[- ]?\d{3}[- ]?\d{3,4}\b"), "{PHONE}"),  # 081-234-5678, 02 123 4567
    # a LINE ID such as @กองคลัง; there is no LINE slot. Not \w: Thai vowel marks are not word characters
    (re.compile(r"@[^\s,()\"']{3,}"), "{URL}"),
]
SLOT = re.compile(r"\{([A-Za-z_]+)\}")


def to_slots(text):
    """Replace real numbers, links and LINE IDs with slots, and unwrap slots the simulator lacks."""
    for pattern, slot in TO_SLOTS:
        text = pattern.sub(slot, text)
    # {LINE_ID} is not a simulator slot: keep the word, drop the braces, so the line still reads
    return SLOT.sub(lambda m: m.group(0) if m.group(1) in SLOT_NAMES else m.group(1), text)


def as_list(value):
    """The model may answer a list or one string; always give a list of non-empty strings."""
    items = value if isinstance(value, list) else str(value or "").split("|")
    return [str(item).strip() for item in items if str(item).strip()]


# the simulator's existing scam scenarios: {scam_type: [scenario dict]}
SCENARIOS = yaml.safe_load(SCENARIOS_YAML.read_text(encoding="utf-8"))["scams"]
SCENARIO_KEYS = {s["key"] for scenarios in SCENARIOS.values() for s in scenarios}


def clean_scenario(item):
    """Keep only allowed values for one post's scenario answer, with slots in place of numbers."""
    out = {}
    for field in ["summary", "caller_role", "goal"]:
        out[field] = to_slots(str(item.get(field) or "").strip())
    out["tactics"] = LIST_SEP.join(to_slots(t) for t in as_list(item.get("tactics")))
    out["phrases"] = LIST_SEP.join(to_slots(p) for p in as_list(item.get("phrases")))

    suitable = str(item.get("call_suitable") or "").strip().lower()
    out["call_suitable"] = suitable if suitable in SUITABLE else ""

    key = str(item.get("scenario_key") or "").strip()
    out["scenario_key"] = key if key in SCENARIO_KEYS else "new"
    group = re.sub(r"[^a-z0-9_]+", "_", str(item.get("scenario_group") or "").strip().lower()).strip("_")
    out["scenario_group"] = out["scenario_key"] if out["scenario_key"] != "new" else group
    return {field: out[field] for field in SCENARIO_FIELDS}


def scenario_list_text():
    """The existing scenarios as prompt text, one line each: type, key, role and goal."""
    lines = []
    for scam_type, scenarios in SCENARIOS.items():
        for s in scenarios:
            lines.append(f"- {scam_type} / {s['key']}: {s['role']}; wants: {' '.join(s['goal'].split())}")
    return "\n".join(lines)


SCENARIO_TASK = Task(
    name="scenarios",
    system_prompt=fill(SCENARIO_SYSTEM, scenarios=scenario_list_text(),
                       slots=" ".join("{" + s + "}" for s in SLOT_NAMES)),
    user_prompt=SCENARIO_USER,
    columns=["post_id", "scam_type", "ORG", "title", "content"],
    fields=SCENARIO_FIELDS,
    clean=clean_scenario,
    tokens_per_post=700,  # eight fields, two of them lists
    batch_size=20,
)


def add_scenario_columns(df):
    """Run the scenario pass on the posts that have a scam_type and add its columns to df.
    Posts with no scam_type get empty scenario fields. Also adds source_url (every post)."""
    scams = df[df["scam_type"] != ""]
    labels = label_all(scams, SCENARIO_TASK)
    for field in SCENARIO_FIELDS:
        df[field] = ""
        df.loc[scams.index, field] = labels[field]
    # a post whose method_channel lists the phone is a phone scam, whatever the second pass said
    phone = (df["scam_type"] != "") & df["method_channel"].str.contains("โทรศัพท์")
    df.loc[phone, "call_suitable"] = "call"
    df["source_url"] = [POST_URL.format(post_id) for post_id in df["post_id"]]
    print(f"{labels['llm_error'].sum()} of {len(scams)} scam posts failed to label")
    return df

In [ ]:
# 20 posts per API call, only posts with a scam_type; adds the scenario columns and source_url
# (a run that stops early resumes from data/llm_scenarios_checkpoint.jsonl)
df = add_scenario_columns(df)
df["call_suitable"].replace("", "(no scam)").value_counts()

In [ ]:
print(f"{len(raw_posts)} posts under #{BASE_TAG} -> {len(df)} kept")
display(df["hashtag"].value_counts())
df["Channel"].value_counts()

In [ ]:
def save_dataset(df, path):
    """Save the dataset as UTF-8 with BOM (so Excel shows the Thai text), dates as DD/MM/YYYY."""
    if pd.api.types.is_datetime64_any_dtype(df["date"]):  # the dataframe keeps dates as datetimes
        df = df.assign(date=df["date"].dt.strftime("%d/%m/%Y"))
    df.to_csv(path, index=False, encoding="utf-8-sig")


# the data was checked right after scraping (see the cell after the scrape)
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
# writes OUT_PATH: data/antifakenews_scam.csv, or ..._test.csv on a test run
save_dataset(df, OUT_PATH)
print(f"saved {len(df)} posts to {OUT_PATH}")

# read it back with the same format so the column is datetime again and sortable
# keep_default_na=False so empty labels show as "" instead of NaN
pd.read_csv(OUT_PATH, parse_dates=["date"], date_format="%d/%m/%Y", keep_default_na=False).head()

In [ ]:
# only the lottery and romance posts, with the columns the call simulator uses
SIMULATOR_COLUMNS = [
    "scam_type", "call_suitable",                              # choosing which posts to use
    "date", "source_url", "summary", "tactics", "phrases",     # the wiki note
    "caller_role", "ORG", "goal", "scenario_group",            # the role card
]
lottery_romance = df[df["scam_type"].isin(["lottery", "romance"])][SIMULATOR_COLUMNS]
print(f"{len(lottery_romance)} posts: {lottery_romance['scam_type'].value_counts().to_dict()}")
lottery_romance